# Chapter 19 – Lab, Parts A and B (ToOp)

Parts A and B of the [lab](../index.md#lab) need ToOp: open this notebook in the ToOp container or with
ToOp's virtual environment, from this folder (`src/`). Each step is a Markdown cell with guidance followed
by an empty cell for your work. The modules next to this notebook do every step once: `security57_toop.py`
(Part A), `ac_n1_toop.py` (Part B) and `backend_compare.py` (Part B, step 5, which needs no ToOp); use them
as building blocks or as references. The written answers are in [solutions.md](../solutions.md#lab-part-a-dc-metrics-from-the-full-n-1-matrix).
Part C is in [lab.ipynb](lab.ipynb).

In [ ]:
import functools  # noqa: F401  (imported for your work below)
from pathlib import Path

import ac_n1_toop  # noqa: F401
import jax
import metrics  # noqa: F401
import numpy as np  # noqa: F401
import security57_toop  # noqa: F401
from toop_engine_dc_solver.jax import default_topology, run_solver_symmetric  # noqa: F401
from toop_engine_dc_solver.jax.aggregate_results import aggregate_to_metric  # noqa: F401
from toop_engine_dc_solver.jax.types import ActionIndexComputations  # noqa: F401

jax.config.update('jax_enable_x64', True)
folder = Path('case57')  # where the IEEE 57 example is written; any writable folder will do
print('ToOp importable; the example grid goes to', folder.resolve())

## Part A, step 1: load IEEE 57

`security57_toop.load(folder)` calls `case57_data_powsybl(folder)`, `load_grid(...)` and `run_initial_loadflow`
and returns `stats`, the static information and the unsplit `(overload_energy_n_0, overload_energy_n_1)`.
Record the stats fields (`n_branches`, `n_nminus1_cases`, `n_relevant_subs`, `n_actions`,
`has_double_limits`) and the two energies. Expect about fifteen seconds.

In [ ]:
# your work here

## Part A, step 2: the full matrices

Build `default_topology(si.solver_config, batch_size=4)` and call `run_solver_symmetric` with a pass-through
aggregation function (`security57_toop.passthrough`, defined once at module level). Check the shapes of the N-0
and N-1 matrices and that every success flag is `True`.

In [ ]:
# your work here

## Part A, step 3: overload energies in NumPy

From `n_0`, `n_1` and `security57_toop.limits(si)` (`max_mw_flow`, `max_mw_flow_n_1`, `overload_weight`),
recompute both energies with `metrics.overload_energy_n1(..., weight=...)` and match ToOp to at least six
digits. Which branches carry the weight 2.0?

In [ ]:
# your work here

## Part A, step 4: double limits

Recompute `max_mw_flow_limited` and `max_mw_flow_n_1_limited` with `metrics.double_limits` and compare with
the arrays in `branch_limits`. Plot, with matplotlib, the relative loading of each branch (worst unsplit
flow / rating) against the ratio limited / rating: three regimes should appear (`security57_toop.regimes`).

In [ ]:
# your work here

## Part A, step 5: real actions and `aggregate_to_metric`

Put action indices into the first slot of two topologies (`security57_toop.with_actions(si, {1: 1, 2: 5})`) and
compute the power-law overload energy, the critical branch count and `n0_n1_delta` for all four topologies
in NumPy. Check each value with `functools.partial(aggregate_to_metric, branch_limits=..., reassignment_distance=...,
n_relevant_subs=..., metric=...)` as the aggregation function, building each partial once
(`security57_toop.toop_metric`). Why is `n0_n1_delta` zero for every topology of this case?

In [ ]:
# your work here

## Part B, step 1: the N-1 definition

Load `data/grid_node_breaker/grid.xiidm` (`ac_n1_toop.default_grid_path()`) with `pypowsybl.network.load`
and build an `Nminus1Definition` by hand: one `MonitoredElement` per line, transformer and bus, one
`Contingency` per branch and a `BASECASE` contingency with no elements. `ac_n1_toop.n1_definition(net)` is
a reference.

In [ ]:
# your work here

## Part B, step 2: run the AC N-1

`get_ac_loadflow_results(net, definition, job_id='lab', timestep=0, n_processes=1)` returns polars
LazyFrames; collect `branch_results`, `node_results`, `va_diff_results` and `converged` into pandas
(`ac_n1_toop.frames`).

In [ ]:
# your work here

## Part B, step 3: metrics with pandas

Recompute `overload_energy_n_1` (field `p`), `overload_current_n_1` (field `i`), `critical_branch_count_n_1`
and `max_va_diff_n_1` with group-bys (the limit of a row is |p| / loading) and compare with
`compute_metrics(results, base_case_id='BASECASE')`.

In [ ]:
# your work here

## Part B, step 4: the row count

Compare the number of branch result rows with
$(2 n_\text{lines} + 2 n_\text{trafos} + 3 n_\text{trafo3w}) \cdot n_\text{contingencies}$
(`ac_n1_toop.expected_rows`). Which rows are missing, and why?

In [ ]:
# your work here

## Part B, step 5 (optional): pandapower against pypowsybl on IEEE 14

`backend_compare.compare()` exports pandapower's `case14` to MATPOWER, loads it in pypowsybl, runs the same
single-branch N-1 in both engines and returns both flow tables and the absolute differences. Plot their
distribution (a histogram of `diff`) and explain the remaining differences. This step needs no ToOp.

In [ ]:
# your work here